In [40]:
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36",
]
random.choice(USER_AGENTS)


'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15'

In [1]:
import pandas as pd
import io


with open("cache/cache.csv", "rb") as f:
    df = io.BytesIO(f.read())

df = pd.read_csv(df.seek(0))

: 

In [ ]:
from pathlib import Path
import pandas as pd

dir = Path('cache/')
p = dir / 'cache.csv'

'cache\\cache.csv'

In [5]:
import sqlite3
import pandas as pd

# Let's say you have scanner results in a DataFrame
df = pd.DataFrame({
    "ticker": ["NVDA", "AMD"],
    "date": ["2026-07-16", "2026-07-16"],
    "close": [125.40, 170.20],
    "volume": [35000000, 18000000],
    "pct_change": [2.10, -1.05]
})

# Connect to database
conn = sqlite3.connect("cache/market_data.db")

# Append DataFrame to the 'stock_history' table
# 'if_exists="append"' tells pandas to add new rows without overwriting the table
df.to_sql("stock_history", conn, if_exists="append", index=False)

conn.close()
print("Data appended successfully using Pandas!")

Data appended successfully using Pandas!


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("cache/market_data.db")

# 2. Read an entire table into a DataFrame
df = pd.read_sql_query("SELECT * FROM stock_history", conn)
df

In [ ]:
from tqdm.notebook import tqdm
import requests
import random
import json
import time
import pandas as pd

TV_SCANNER_URL = "https://scanner.tradingview.com/global/scan"
TV_USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36",
]
CLOSE_TV_COLUMNS = [
    "name",
    "change",
    "premarket_gap",
    "close",
    "high",
    "low",
    "volume",
]

def get_close_infos(tickers: list[dict], batch_size: int = 500) -> list:
    results = []
    for i in tqdm(range(0, len(tickers), batch_size), desc="Fetching batches", unit="batch"):
        batch = tickers[i:i + batch_size]
        tv_payload = {
            "symbols": {
                "tickers": [t["tv_ticker"] for t in batch],
                "query": {"types": []}
            },
            "columns": CLOSE_TV_COLUMNS
        }
        tv_headers = {
            "Content-Type": "application/json",
            "User-Agent": random.choice(TV_USER_AGENTS)
        }
        try:
            tv_response = requests.post(
                TV_SCANNER_URL, headers=tv_headers, data=json.dumps(tv_payload))
            tv_response.raise_for_status()
            tv_data = tv_response.json()
        except Exception as e:
            print(
                f"TV scanner request failed for batch {i // batch_size + 1}: {e}")
            continue
        for _item in tqdm(tv_data.get('data', []), desc=f" > Fetching batch {i // batch_size + 1}", unit="ticker", leave=False):
            item = _item['d']

            row = {
                "ticker": item[CLOSE_TV_COLUMNS.index("name")],
                "change": item[CLOSE_TV_COLUMNS.index("change")],
                "premarket_gap": item[CLOSE_TV_COLUMNS.index("premarket_gap")],
                "close": item[CLOSE_TV_COLUMNS.index("close")],
                "high": item[CLOSE_TV_COLUMNS.index("high")],
                "low": item[CLOSE_TV_COLUMNS.index("low")],
                "volume": item[CLOSE_TV_COLUMNS.index("volume")],
            }
            results.append(row)
        time.sleep(0.25)
    return results

In [47]:

from module.st import get_tickers, is_trading_day, get_close_infos
import sqlite3
from sqlite3 import Connection
import pandas as pd
from datetime import datetime

def is_fetched(conn: Connection, table_name: str = "st") -> bool:
    return pd.read_sql_query(f"select max(date) as latest_date from {table_name}", conn).iloc[0].latest_date == datetime.now().strftime("%Y-%m-%d")

def fetch_ticker_names(conn: Connection, table_name: str = "t"):
    print("fetching ticker names...")
    fetched_tickers = get_tickers(min_cap=2_000_000_000, max_results=2_000, avg_daily_vol=2_000_000)
    fetched_tickers = pd.DataFrame(fetched_tickers)
    old_tickers = pd.read_sql_query(f"select * from {table_name}", conn)
    new_tickers = fetched_tickers[~fetched_tickers['tv_ticker'].isin(old_tickers['tv_ticker'])]
    new_tickers.to_sql(table_name, conn, if_exists="append", index=False)

def fetch_ticker_infos(conn: Connection, ticker_table_name: str = "t", info_table_name: str = "st"):
    print("fetching ticker infos...")
    tickers = pd.read_sql_query(f"select * from {ticker_table_name}", conn)
    tickers = tickers.to_dict(orient='records')
    ticker_infos = get_close_infos(tickers=tickers)
    ticker_infos = pd.DataFrame(ticker_infos)
    ticker_infos['date'] = datetime.now().strftime("%Y-%m-%d")
    ticker_infos.to_sql(info_table_name, conn, if_exists="append", index=False)


In [10]:
conn = sqlite3.connect("cache/data.db")

In [51]:
# tickers = get_tickers(min_cap=2_000_000_000, max_results=2_000, avg_daily_vol=2_000_000)
# tickers = pd.DataFrame(tickers)
# tickers.to_sql("t", conn, if_exists="append", index=False)

# fetch_ticker_names(conn=conn)
# tickers = pd.read_sql_query("select * from t", conn)
# tickers = tickers.to_dict(orient='records')
# tickers

# fetch_ticker_infos(conn=conn)

# infos = pd.read_sql_query("select * from st", conn)
# infos

pd.read_sql_query("select * from st where ticker = 'NVDA'", conn)

,ticker,change,premarket_gap,close,high,low,volume,date
0,NVDA,0.3305,-0.823529,212.5,213.81,206.040,124796251,2026-07-16
1,NVDA,-2.4000,-0.823529,207.4,211.08,205.845,122984681,2026-07-17


In [2]:
tickers = fetch_ticker_infos(conn=conn)
save_sqlite(tickers, conn=conn)
conn.close()

Fetching batches: 100%|██████████| 2/2 [00:02<00:00,  1.09s/batch]

data saved to cache/data.db


In [24]:
conn = sqlite3.connect("cache/data.db")
df = pd.read_sql_query("select max(date) as latest_date from st", conn).iloc[0].latest_date == datetime.now().strftime("%Y-%m-%d")
df

False

In [32]:
conn.close()